# LangChain으로 만드는 웹 기반 챗봇 실습  

##  1. 과정 소개 및 개요
- **학습 목표**
  - AI 챗봇의 기본 개념과 발전 과정을 이해한다.
  - LangChain, Streamlit, FAISS 등 주요 기술의 역할과 장점을 이해한다.
  - LCEL을 사용하여 확장성 있는 체인을 직접 설계할 수 있다.
  - **웹 기반으로 자신이 커스텀한 챗봇을 배포할 수 있는 역량을 갖춘다.**

- **핵심 키워드**
  - LLM Chatbot, LangChain, LCEL, Prompt, Streamlit, FAISS, RAG  

### 환경 설정 

#### (1) 필수 라이브러리 설치 
웹 페이지(UI) 를 직접 띄워야 하기 때문에, Google Colab보다는  
본인 PC의 로컬 환경(VSCode, Terminal 등) 에서 실행하는 것을 권장합니다.

먼저 실습에 필요한 주요 패키지를 설치합니다.
터미널 또는 VSCode의 터미널 창에서 다음 명령어를 입력하세요.
```bash
pip install -U langchain langchain-openai streamlit faiss-cpu python-dotenv tiktoken
```


#### (2) 라이브러리별 역할 설명

| 라이브러리                | 주요 역할        | 설명                                                                                                  |
| -------------------- | ------------ | --------------------------------------------------------------------------------------------------- |
| **langchain**        | 체인 프레임워크 핵심  | LLM(대형언어모델)을 중심으로 다양한 입력·출력·도구를 연결하는 프레임워크. `PromptTemplate`, `LLMChain`, `Retriever`, `LCEL` 등을 제공 |
| **langchain-openai** | OpenAI 연동 모듈 | OpenAI의 GPT 모델(`ChatOpenAI`)을 LangChain 체인 안에서 바로 사용할 수 있도록 연결해줌                                    |
| **streamlit**        | 웹 인터페이스 구현   | Python 코드 몇 줄로 대화형 웹 UI(챗봇 화면, 입력창, 출력창 등)를 만들 수 있는 간단한 프레임워크                                       |
| **faiss-cpu**        | 벡터 검색 DB     | 문서 검색 기반 챗봇(RAG) 구현 시, 텍스트 임베딩을 벡터로 저장하고 유사한 내용을 빠르게 검색하는 역할                                        |
| **python-dotenv**    | 환경변수 관리      | `.env` 파일에서 OpenAI API Key 등 민감한 정보를 안전하게 불러와 코드에 직접 노출하지 않게 해줌                                     |

#### (3) OpenAI API 키 설정 (.env 파일, 보안 주의)

**API란?**

하나의 프로그램이 다른 프로그램이나 시스템과 **데이터를 주고받고 기능을 활용할 수 있도록 연결해주는 인터페이스**를 의미합니다.  
즉, **서로 다른 소프트웨어 간의 다리 역할을 하는 통신 규칙**입니다.

예를 들어, OpenAI의 **GPT API**는 사용자가 직접 모델을 설치하지 않아도  
OpenAI 서버에 **질문(요청)** 을 보내고, 그 결과로 **답변(응답)** 을 받을 수 있게 해주는 API입니다.

→ 쉽게 말해, **필요할 때마다 GPT에게 요청을 보내 답변을 받아오는 온라인 시스템**이라고 생각하시면 됩니다.

※ 이 API는 **사용량 기반으로 과금되는 유료 서비스**이며, 웹에서 제공되는  
ChatGPT의 무료·유료 플랜과는 **별도의 요금 체계**로 운영됩니다.


※ 현재 사용 가능한 대표 GPT API 모델 일부 (2026년 3월 기준)

| 모델 이름 | 설명 | 입력 가격 (per 1M tokens) | 출력 가격 (per 1M tokens) | 추천 용도 |
|-----------|--------|---------------------------:|----------------------------:|-----------|
| `gpt-3.5-turbo` | 초창기 레거시 모델 | **$0.5** | **$1.5** | 기존 예제 호환, 구형 자료 참고용 |
| `gpt-4o-mini` | 빠르고 저렴한 범용 소형 모델 | **$0.30** | **$1.20** | 가벼운 멀티모달 앱, FAQ 챗봇, 분류/추출, 파인튜닝·증류용 |
| `gpt-4.1-mini` | 성능, 속도, 비용 측면에서 균형이 좋은 모델 | **$0.8** | **$3.2** | 지시 이행, 툴 호출, 긴 컨텍스트 처리, 실무형 API 앱 |
| `gpt-4.1-nano` | 더 가볍고 저렴한 초경량 모델 | **$0.2** | **$0.8** | 초저비용 분류, 간단한 자동화, 대량 처리, 저지연 서비스 |
| `o4-mini` | 빠른 추론에 강한 모델 | **$4.0** | **$16.0** | 복잡한 추론, 코딩 보조, 시각 이해, 다단계 문제 해결 |
| `gpt-5-mini` | 빠르고 효율적인 GPT-5 계열 모델 | **$0.25** | **$2.0** | 저지연·고볼륨 실서비스, 명확한 업무형 프롬프트, 범용 실습 |
| `gpt-5-nano` | GPT-5 계열 최저가 모델 | **$0.05** | **$0.4** | 요약, 분류, 태깅, 대량 배치 처리, 입문 실습/초저비용 API |

※ OpenAI는 지속적으로 모델명을 업데이트하고 있으므로, [공식 문서](https://platform.openai.com/docs/models)에서 최신 목록을 확인하는 것이 좋습니다.

최신 요금은 OpenAI 공식 문서에서 확인하세요:  
https://openai.com/api/pricing/

**API Key란?**  
GPT API를 사용하기 위해서는 OpenAI 서버가 요청을 보낸 사용자를 식별하고,  
해당 사용자의 **이용량과 과금 내역을 추적**할 수 있어야 합니다.  
이를 위해 OpenAI는 각 사용자에게 **고유한 인증 토큰(API Key)** 을 발급합니다.

이 키는 **요금 청구의 기준이 되는 중요 정보**이며, 외부에 노출될 경우 **타인이 임의로  
사용해 요금이 발생할 수 있으므로 철저한 보안 관리가 필요**합니다.


**발급 절차**

1. [https://platform.openai.com/](https://platform.openai.com/) 접속  
2. 로그인 또는 계정 신규 생성  
3. 우측 상단 프로필 아이콘 클릭 → `View API Keys` 선택  
4. 결제 수단(신용카드 등) 등록  
5. `+ Create new secret key` 클릭 → **발급된 Key를 복사해 안전한 위치에 보관**

**발급된 Key는 한 번만 표시되므로**, 반드시 즉시 메모장이나 `.env` 파일 등에 백업해 두세요.

실습 폴더 루트에 .env 파일을 생성하고, 강사가 안내한 OpenAI API Key를 아래 형식으로 저장합니다. 
```bash
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
``` 

#### ⚠️ 주의
.env 파일은 절대 깃허브(GitHub)나 외부 저장소에 업로드하지 마세요.  
사용은 `.env` 파일에 저장 후 `python-dotenv`를 이용해 불러오는 방식을 권장합니다.
API Key가 노출되면 과금 위험이 있습니다.

```python
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
```

##  2. LangChain 소개
### 🔹 LangChain이란?

<img src ="image/LangChain.png" width="500">

LangChain은 **대규모 언어모델(LLM)** 을 더 쉽게 활용할 수 있도록 도와주는 **프레임워크**입니다.  

단순히 LLM에게 질문을 던지는 수준을 넘어서, **프롬프트 관리(prompt management)**,  
**메모리(memory)**, **외부 도구(tool) 연동**, **문서 검색(RAG)** 등을 체계적으로 구성할 수 있습니다.

예를 들어, 우리가 ChatGPT 같은 LLM을 사용할 때  
단순히 모델 하나만으로 뛰어난 답변이 만들어지는 것은 아닙니다.  

실제로는 **검색(Search)**, **분석(Analysis)**, **정보 정리(Summarization)** 등  
여러 절차가 백그라운드에서 함께 작동해야 더 정확하고 풍부한 답변을 얻을 수 있습니다.  

LangChain은 이러한 여러 단계를 **하나의 체계적인 구조로 묶어주는 프레임워크**입니다.  
즉, LLM이 단독으로 모든 일을 처리하는 대신  
“사용자 입력 → 관련 정보 검색 → 분석 및 요약 → 결과 생성”과 같은  
복잡한 흐름을 하나의 **체인(chain)** 으로 자동화할 수 있습니다.  

<img src="image/chatgpt.jpg" width="600">

예를 들어, 사용자가 “3박 4일, 30만 원 예산, 맛집 중심의 서울 여행”을 요청하면,  
LangChain은 내부적으로 다음과 같은 과정을 수행합니다.  

> 입력 분석 → 예산과 일정 파악 → 추천 장소 검색 → 일정표 생성 → 자연스러운 문장으로 요약  

이처럼 LangChain은 단순한 대화 요청을 넘어 **검색, 분석, 조합, 요약** 등의 단계를 유기적으로 연결해  
LLM이 더 똑똑하게 작동하도록 돕습니다.  

또한 LangChain은 **LCEL (LangChain Expression Language)** 이라는 표현 언어를 통해  
이러한 여러 단계(components)를 **직관적으로 연결**할 수 있게 해줍니다.  
즉, “LLM → 요약기 → 출력 포맷터”처럼 **함수형 파이프라인**을 선언적으로 작성할 수 있어  
확장성과 유지보수성이 뛰어납니다.

LCEL 예시: 
```text
사용자입력
  | 분석기(요청에서 날짜·예산·키워드 추출)
  | 검색기(분석 결과를 바탕으로 추천 장소 탐색)
  | 일정생성기(추천 장소로 일자별 일정 구성)
  | 요약기(자연스러운 문장으로 결과 정리)
  | 출력
```

이처럼 LCEL은 각 단계를 함수처럼 “연결(pipe)”하여 데이터가 흘러가는 경로를 선언적으로 표현합니다.  
복잡한 제어문 없이 “입력 → 처리 → 출력” 구조를 직관적으로 구성할 수 있어,
체인 전체의 논리 흐름을 한눈에 파악할 수 있습니다.

### 🔹 간단한 예시 코드

In [2]:
# OpenAI API를 LangChain에서 쉽게 쓸 수 있도록 해주는 래퍼 모듈
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import PromptTemplate # 문자열 템플릿 기반으로 프롬프트를 자동 구성해주는 유틸
from langchain_core.output_parsers import StrOutputParser # LLM의 응답 객체를 문자열로 변환해주는 파서 (출력 후처리 단계)
from dotenv import load_dotenv # .env 파일의 환경변수를 자동으로 불러오기 위한 모듈
load_dotenv()  # 실행 시 .env 파일을 찾아 변수들을 환경에 로드

# LCEL 방식: 파이프라인 형태로 선언
llm = ChatOpenAI(model="gpt-5-nano")
prompt = PromptTemplate.from_template("'{topic}' 주제에 대해 한 문장으로 설명해줘.")
output_str = StrOutputParser() # 결과 객체에서 텍스트(content)만 깔끔하게 추출해 문자열로 변환함

# LCEL 표현 (프롬프트 → LLM → 출력)
chain = prompt | llm | output_str

# chain.invoke() 는 전체 파이프라인을 한 번 실행하는 메서드
result = chain.invoke({"topic": "framework"})
print(result)

프레임워크는 특정 유형의 애플리케이션을 빠르고 일관되게 개발할 수 있도록 재사용 가능한 구조, 규칙, 도구를 미리 제공하는 뼈대이다.


| 항목            | 설명                              |
| ------------- | ------------------------------- |
| **LangChain** | LLM을 활용하기 위한 파이썬 기반 프레임워크       |
| **핵심 기능**     | 프롬프트 관리, 메모리, 도구 연동, 문서 검색(RAG) |
| **LCEL**      | 체인 구성 과정을 간결하게 표현하는 언어          |
| **활용 예시**     | 질문응답, 요약, 문서검색, 챗봇, 자동화된 지식시스템  |


LangChain은 LLM의 “두뇌”를 실무에서 “자동화 가능한 시스템”으로 바꿔주는 연결 프레임워크입니다.  
단순한 모델 호출이 아니라, 프로세스를 설계하고 실행하는 엔진이라고 이해하면 됩니다.

## 3. 프롬프트(Prompt)

**프롬프트(Prompt)** 는 LLM(대형언어모델)에게 “무엇을, 어떻게 해달라”고 **지시하는 문장**입니다.

사람에게 질문할 때도 요청을 명확히 해야 원하는 답을 얻을 수 있듯,  LLM에게도 프롬프트가 곧 **“명령어이자 대화의 시작점”** 이 됩니다.  

예를 들어,  
> 1. “고양이에 대해 설명해줘.”  
> 2. “고양이를 5살 아이가 이해할 수 있게 설명해줘.”  

두 문장은 같은 주제지만 결과는 완전히 다릅니다.  
👉 따라서 **프롬프트의 설계(=프롬프트 엔지니어링)** 가 답변의 질을 결정합니다.

#### LLM은 세 가지 종류의 프롬프트로 대화한다
LLM은 단순히 한 문장만 보고 답하지 않습니다.  
대화의 **맥락(context)** 을 유지하기 위해 다음 세 가지 프롬프트를 함께 사용합니다.

| 종류 | 역할 | 예시 |
|------|------|------|
| **System Prompt** | 모델의 성격과 역할을 설정 | “너는 친절하고 이해심 많은 선생님이야.” |
| **User Prompt** | 사용자의 실제 질문이나 요청 | “인공지능이 뭐야?” |
| **Assistant Prompt** | 모델이 이전에 했던 응답 | “인공지능은 사람의 학습 능력을 모방한 기술이야.” |

#### 왜 이렇게 구분할까?
- **System Prompt** → 모델의 **기본 성격**을 결정합니다. (친절한 선생님 / 냉정한 분석가 등)  
- **User Prompt** → 사용자가 **무엇을 알고 싶은지**를 알려줍니다.  
- **Assistant Prompt** → 이전 대화 내용을 **기억하고 맥락을 유지**하게 도와줍니다.  

이 세 가지를 함께 쓰면 모델은  
> “지금 나는 친절한 선생님이고, 방금 ‘인공지능이 뭐냐’는 질문을 들었구나.”  
라고 이해한 뒤 더 자연스럽고 일관된 답변을 합니다.

### 🔹 LangChain에서의 프롬프트: `PromptTemplate`
LangChain에서는 프롬프트를 **하드코딩(직접 작성)** 하는 대신, **`PromptTemplate`** 클래스를 사용해 “템플릿” 형태로 관리합니다.

#### 예시 1

In [5]:
from langchain_core.prompts  import PromptTemplate

# {topic} 부분에 사용자의 입력이 들어가도록 템플릿 생성
prompt = PromptTemplate.from_template("'{topic}'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘.")

# 실제 실행 시:
prompt.format(topic="framework")
# 결과 → "'LangChain'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘."

"'framework'에 대해 초보자도 이해할 수 있게 한 문장으로 설명해줘."

#### 예시 2

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1.  모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2️. 프롬프트 구성
# - System: 모델의 기본 역할(여행 전문가)
# - User: 사용자의 첫 질문 (가을 여행지 추천)
# - Assistant: 직전에 모델이 대답했던 내용 (AI의 응답)
# - User: 현재 사용자의 새로운 질문 (부산 일정표 요청)
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 친절한 여행 전문가야. 사용자의 취향에 맞게 튀르키예 여행지를 추천하고, 일정도 구체적으로 제안해줘."),
    ("user", "가을에 12박 13일로 갈만한 튀르키예 여행지를 추천해줘."),
    ("assistant", "이스탄불, 카파도키야는 꼭 포함해줘! 각각 바다와 음식, 분위기가 다르니 다양하게 포함해줘."),
    ("user", "그럼 이번엔 튀르키예로 10박 12일 일정 만들어줘.")
])

# 3️. 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# 4️. LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 5️. 실행
result = chain.invoke({})

print(result)

좋아요! 10박 12일로 튀르키예를 알차게 돌아보는 루트를 정리해볼게요. 아래 계획은 주요 도시 5곳(이스탄불, 카파도키아, 이즈미르/에페소스, 파묵칼레, 안탈리아)을 효율적으로 잇는 버전입니다. 마지막에 11박 12일 버전도 간단히 대안으로 적어둘게요.

10박 12일 버전 (10박으로 12일 간의 여유로운 페이스)
- Day 1: 이스탄불 도착. 스팔란사타 지역의 숙소에 체크인 후 구시가지 산책(소피아 모스크, 블루 모스크, 카프즐루 cistern 등 근처 짧은 코스). 
- Day 2: 이스탄불 구시가지 집중 탐방. 아야 소피아 박물관, 톱하파샤 궁전, 그랜드 바자 등. 저녁은 보스포루스 해협 크루즈 추천.
- Day 3: 이스탄불의 추가 명소 (돌마바흐체 궁전, Spice Bazaar 방문) 또는 이스탄불의 활기 있는 한블록인 타르크(갈라타) 지역 산책. 야간 비행으로 카파도키아 이동 준비.
- Day 4: 이스탄불 → 카파도키아 도착. 고레메 기행지(동굴 마을) 체크인. 저녁은 바위 언덕에서 전망 즐기기.
- Day 5: 카파도키아 풀코스 하루(열기구는 선택사항). 고레메 야외박물관, 파사바그, 데브렌트 밸리 등 탐방.
- Day 6: 카파도키아에서 이즈미르로 이동(일정에 따라 카멜리나 Denizli 등 경유 가능). 이즈미르 혹은 셀추크 숙소 체크인. 근처 시내 산책 추천.
- Day 7: 에페소스 고대 도시 방문. 바다마을 셀주크 또는 셰렌크(Şirince) 마을 가볍게 들르기. 이즈미르 숙박.
- Day 8: 이즈미르에서 파묵칼레로 이동. 파묵칼레의 석회암 사문(Hierapolis)과 석순 온천 구경. 파묵칼레 인근 숙박.
- Day 9: 파묵칼레에서 안탈리아로 이동. 안탈리아 구시가지(Kaleiçi) 산책, 밤에는 해안 산책로 추천.
- Day 10: 안탈리아에서 페르게, 아스펜도스 등 근교 유적지 당일치기 투어. 해질녘 케일리치 구시가지 재방문 가능.
- Day 11: 안탈리아에서 이스탄불로 국내선 이동(오전 또는 이른 오후 비행). 이스탄불

#### 예시 3. 퓨샷 러닝(Few-shot Learning)

**Few-shot Learning**은 LLM에게 **“예시 몇 개를 직접 보여줘서 학습 방향을 잡아주는 방법”** 입니다.

AI에게 “이렇게 대답해!”라고 명령하는 것만으로는 부족할 때,  
실제 **입출력 예시를 몇 개 제시해주면** 모델이 그 **패턴과 말투, 구조**를 빠르게 학습해 유사한 답변을 생성합니다.

예를 들어  
> “백종원 대표의 충청도 사투리로 요리 팁을 말해줘.”  
라고만 하면 AI는 사투리의 뉘앙스를 정확히 몰라 평범한 답변을 할 수 있습니다.  

하지만 아래처럼 실제 대화 예시를 1~2개 보여주면  
> “아~ 이런 식으로 말하면 되는구나!”  
하고 패턴을 스스로 학습해 자연스러운 말투를 재현합니다.

**퓨샷 러닝의 장점**

| 장점 | 설명 |
|------|------|
|**말투·스타일 학습 가능** | 특정 인물(예: 백종원, 아이유, 선생님 등)의 말투나 표현방식 재현 |
|**작업 포맷 유지** | 요약, 번역, 해설 등 특정 형식을 예시로 제시하면 그 구조를 그대로 따름 |
|**추가 학습(파인튜닝) 없이 가능** | 단순히 프롬프트 안에 예시 몇 개 넣는 것만으로도 “즉석 학습” 효과 |
|**창의적 제어** | “~처럼 말해줘”, “이런 형식으로 답해줘” 등 자유로운 응용 가능 |


In [5]:
### 코드 예시: 백종원 말투 흉내내기 (Few-shot Learning)
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# temperature를 약간 높이면(0.8) 표현이 더 자유롭고 창의적이게 됨
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.8)

# - System: 모델의 기본 역할 지정
# - 예시 대화(few-shot examples): 실제 말투 패턴을 보여줌
# - User: 새로운 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 요리 전문가이자, 백종원 대표의 말투를 자연스럽게 흉내내는 AI야. 충청도 사투리와 친근한 말투로 대답해."),
    
    # [예시 1] — 사용자가 물어봤고, 백종원 말투로 답변한 사례
    ("user", "라면 맛있게 끓이는 비법 알려주세요."),
    ("assistant", "그라믄 말이여~ 라면은 물 끓일 때 스프를 같이 넣는 게 포인트여. 그래야 국물이 진~해진다잉."),
    
    # [예시 2] — 또 다른 백종원식 대답 예시
    ("user", "김치찌개 맛있게 끓이는 법은요?"),
    ("assistant", "김치찌개는 묵은지 써야혀~ 그래야 깊은 맛이 나. 돼지고기 좀 넣어주면 금상첨화여잉."),
    
    # [새로운 요청] — 실행 시 전달될 질문 변수. 이제 모델이 같은 말투로 이어서 대답해야 함
    ("user", "{question}")])

# 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 실행 — invoke()에 직접 새로운 질문 전달
result = chain.invoke({"question": "김치전은 어떻게 만들면 맛있어요?"})

print(result)

오호~ 김치전 맛있게 굽는 법도 쉽게 해보자. 묵은김치가 포인트여, 그 맛이 확 살아나거든.

재료 (2인분 정도)
- 묵은 김치 2컵 정도(잘게 썰기)
- 부침가루 또는 밀가루+전분 1컵 정도
- 물 1/2컵 ~ 3/4컵(반죽 농도에 맞춰 조절)
- 소금 한 꼬집(필요하면)
- 대파 1대 송송, 양파 조금 채 썰기
- 선택: 돼지고기 얇게 썬 것 100g, 새우, 볶은 해물 등
- 식용유 충분히

맛 팁
- 김칫국물 조금 넣으면 깊은 맛이 남. 반죽에 한 큰술 정도 넣어도 좋다.
- 김치를 살짝 볶아 물기를 좀 날려주면 바삭함이 더 살아난다.
- 반죽은 되도록 덜 섞되, 덩어리 없이 잘 섞이도록.

만드는 법
1) 김치를 먹기 좋게 자르고, 대파, 양파도 준비한다.
2) (선택) 팬에 김치를 1~2분 정도 살짝 볶아 물기를 조금 날려준다.
3) 큰 볼에 부침가루와 물을 넣고 고루 섞되, 너무 묽지 않게 덩어리 없이 섞는다.
4) 다진 김치와 채소, 선택한 고기류를 반죽에 넣고 가볍게 버무린다.
5) 중간 불로 달군 팬에 넉넉히 기름을 두르고 반죽을 얇고 넓게 펼친다.
6) 앞면이 살짝 갈색이 나고 크리스피해질 때까지 2~3분 정도 굽고, 뒤집어서 반대편도 2~3분 더 굽는다.
7) 바삭하게 익으면 종이 타월 위에 잠시 두어 기름기를 뺀다.

곁들임 앙팡
- 간장 소스: 간장 2큰술 + 식초 2큰술 + 설탕 1작은술 + 물 2큰술 + 다진 파 조금 + 고춧가루 약간
- 초고추장 소스나 마늘 찍은 간장도 잘 어울린다.

마무리 팁
- 김치가 신김치면 중간에 고추가루 조금 넣어 매콤함을 조절해도 좋다.
- 얇고 넓게 펴서 바삭하게 구우면 더 맛있다.
- 김치의 수분이 너무 많으면 반죽이 흐물해지니, 물은 천천히 넣고 농도 맞춰 보자.

한번 해봐라~ 이거 하나면 김치전은 집에서도 금방 맛있게 만들어진다!


##  4. Q&A 챗봇 실습 (LCEL)

앞서 배운 **프롬프트 설계**와 **LCEL 파이프라인 구성법**을 직접 활용하여  
“질문 → 답변” 형태의 **Q&A 챗봇**을 완성해보세요.

단순히 모델을 호출하는 것이 아니라, **프롬프트 구조를 설계**하고 **출력 파서를 연결**하며  
 **LCEL 문법(| 연산자)** 을 이용해 **체인형 파이프라인**을 구성해야 합니다.

##### 실습 시나리오
> 당신은 **AI Q&A 챗봇 개발자**입니다.  
> 사용자가 “기술 관련 질문”을 하면, 챗봇은 **요약 + 예시 + 비유 설명**을 포함한 답변을 하도록 만들어야 합니다.  
> (예: “REST API란?”, “클라우드 컴퓨팅이 뭐야?”, “RAG는 어떤 구조야?” 등)

즉, 단순한 정의를 넘어 “무엇인지 + 왜 중요한지 + 예시” 를 한 번에 설명하는 챗봇을 설계하세요.

**과제 요구사항**

1. **`ChatPromptTemplate`** 을 사용해 아래 구조의 프롬프트를 직접 설계하세요.
   - `system`: 챗봇의 역할 정의  
     (예: “너는 기술 개념을 쉬운 비유와 예시를 섞어서 설명하는 AI 선생님이야.”)
   - `user`: 사용자의 질문을 `{question}` 변수로 전달

2. **`ChatOpenAI`** 모델을 연결하세요.  
   - 모델: `"gpt-5-nano"`  
   - `temperature=0.7` (조금 창의적인 답변 유도)

3. **`StrOutputParser`** 를 사용하여 결과를 문자열로 출력하세요.

4. **LCEL 문법**으로 체인을 완성하세요.  
   예:  
   ```python
   chain = prompt | llm | parser
   ```
5. 실행해보고 답변이 지나치게 딱딱하거나 비유나 예시가 부족하면 Prompt를 직접 수정하면서 성능을 조정해보세요.

**질문 답변 예시:**  

질문 : REST API란?  

답변 :   

① **정의**: REST API(Representational State Transfer Application Programming Interface)는 웹 상에서 서로 다른 프로그램이나 시스템이 데이터를 주고받을 수 있도록 하는 규칙과 방법을 정의한 인터페이스입니다. REST는 HTTP 프로토콜을 기반으로 하며, 자원(Resource)을 URI(Uniform Resource Identifier)로 식별하고, 다양한 HTTP 메서드(예: GET, POST, PUT, DELETE)를 사용하여 자원에 대한 작업을 수행합니다.

② **이유(중요성)**: REST API는 서로 다른 시스템 간의 통신을 쉽고 효율적으로 만들어줍니다. 이를 통해 개발자들은 복잡한 시스템을 간단하게 연결하고, 데이터를 주고받을 수 있습니다. REST API는 웹 기반 애플리케이션에서 표준으로 자리잡고 있어, 다양한 서비스를 통합하고 확장하는 데 매우 중요합니다.

③ **쉬운 예시**: REST API를 설명하기 위해 레스토랑을 비유로 들어볼 수 있습니다. 레스토랑의 메뉴판은 고객이 어떤 음식을 주문할 수 있는지를 보여주는 역할을 합니다. 고객(클라이언트)은 메뉴판(URI)을 보고 원하는 음식을 선택하고, 주문(HTTP 메서드)을 합니다. 주방(서버)은 고객의 주문을 받아 음식을 준비하고, 다시 고객에게 서빙합니다. 이 과정에서 메뉴판은 고객과 주방 간의 소통을 원활하게 해주는 역할을 하며, REST API는 시스템 간의 소통을 원활하게 해주는 역할을 합니다.


In [11]:
### 여기에 과제 요구 사항을 작성하세요
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# temperature를 약간 높이면(0.8) 표현이 더 자유롭고 창의적이게 됨
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.8)

# - System: 모델의 기본 역할 지정
# - 예시 대화(few-shot examples): 실제 말투 패턴을 보여줌
# - User: 새로운 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 **AI Q&A 챗봇 개발자**입니다."),

#요약 + 예시 + 비유 설명
    ("user", "REST API란?"),
    ("assistant", "REST API는 HTTP를 이용해 자원을 CRUD 방식으로 다루는 API 설계 방식입니다. "
              "마치 식당에서 메뉴판(URL)을 보고 주문(HTTP 메서드)하는 것과 비슷해요. "
              "예를 들어 GET /users/123은 '123번 사용자 정보를 조회해줘'라는 뜻이에요."),

("user", "{question}")])

# 출력 파서 (응답 텍스트만 추출)
parser = StrOutputParser()

# LCEL 파이프라인 구성 (Prompt → LLM → Parser)
chain = prompt | llm | parser

# 실행 — invoke()에 직접 새로운 질문 전달
result = chain.invoke({"question": "클라우드 컴퓨팅이 뭐야?"})

print(result)

클라우드 컴퓨팅은 인터넷을 통해 필요할 때마다 서버, 저장소, 데이터베이스, 소프트웨어 등 IT 자원을 제공받아 사용하는 서비스 형태를 말합니다. 직접 하드웨어를 소유하거나 관리하지 않아도 되고, 사용량에 따라 비용을 지불하는 것이 일반적입니다.

주요 특징
- 온디맨드 셀프서비스: 필요하면 바로 자원을 사용할 수 있습니다.
- 광범위한 네트워크 접근: 인터넷으로 어디서나 접근 가능.
- 리소스 풀링과 다중 임대: 여러 사용자가 자원을 공유하되 격리되어 사용합니다.
- 빠른 탄력성: 필요에 따라 자원을 즉시 늘리거나 줄일 수 있습니다.
- 측정 가능한 서비스: 사용량을 모니터링하고 요금으로 청구합니다.

서비스 모델
- IaaS(Infrastructure as a Service): 서버/스토리지/네트워크 같은 기본 인프라를 제공
- PaaS(Platform as a Service): 개발/운영 플랫폼을 제공해 애플리케이션 개발이 쉬움
- SaaS(Software as a Service): 완성된 소프트웨어를 구독 형태로 제공

배포 모델
- 퍼블릭 클라우드: 다수의 고객이 같은 인프라를 공유
- 프라이빗 클라우드: 한 기업 전용으로 운영
- 하이브리드 클라우드: 퍼블릭과 프라이빗을 조합
- 멀티 클라우드: 여러 클라우드 공급자를 함께 사용하는 방식

장점
- 비용 효율성: 초기 대규모 투자 없이 시작 가능
- 확장성/탄력성: 수요 변화에 따라 자원 자동 조정
- 운영 간소화: 관리 effort 감소, 자동화 가능
- 빠른 배포: 새로운 서비스나 애플리케이션을 신속히 출시

주의점
- 보안/규정 준수 및 데이터 주권 문제
- 벤더 종속성(종속성 관리 필요)
- 네트워크 의존성 및 성능 예측의 어려움
- 데이터 이관/백업 전략 필요

예시
- 퍼블릭 IaaS/PaaS/SaaS 제공자: AWS, Microsoft Azure, Google Cloud
- 예시 소프트웨어: Gmail, Salesforce, Slack( SaaS 형태)

간단히 말해, 클라우드 컴퓨팅은 

<details>
<summary>정답 보기</summary>

```python 
from langchain_openai import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7,  max_tokens=200)

# 프롬프트 설계
# system: 챗봇의 역할 정의
# user: {question} 변수를 통해 질문 전달
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "너는 기술 개념을 알기 쉽게 설명하는 AI 선생님이야. "
     "답변은 반드시 ① 정의 ② 이유(중요성) ③ 쉬운 예시를 포함해야 해. "
     "필요하다면 비유를 사용해도 좋아."),
    ("user", "{question}")
])

# 출력 파서
parser = StrOutputParser()

# LCEL 체인 구성
chain = prompt | llm | parser

# 실행 테스트
question = "REST API란?"
answer = chain.invoke({"question": question})

print(f"🧠 질문: {question}\n")
print(f"💬 답변:\n{answer}")
```
</details>

##  5. 대화형 챗봇 (메모리)

### 🔹 Conversation Memory란?
> **Conversation Memory(대화 메모리)** 는 LLM이 **이전 대화 내용을 기억하고 이어서 대답**할 수 있도록 하는 기능입니다.

일반적인 LLM은 한 번의 요청(prompt)만 처리하고 끝나기 때문에,  
사용자가 “아까 말한 여행지 일정 다시 알려줘.”처럼 과거 대화를 참조하면 맥락을 잃어버립니다.  
→ 이때 필요한 것이 바로 **Memory(기억 기능)** 입니다.

### 🔹 왜 Memory가 필요할까?

| 상황 | Memory 없을 때 | Memory 있을 때 |
|------|----------------|----------------|
| 사용자: “부산 여행지 추천해줘.”<br>→ 모델이 답변함 | 다음 질문: “그럼 거기 근처 맛집은?”<br>→ 모델이 “어디 여행 말씀인가요?” | “부산 근처엔 회센터가 많고… 홍합탕집이 유명해요!” |
| 사용자: “어제 추천해준 책 제목 다시 말해줘.” | “어떤 책을 말씀하시는 건가요?” | “어제 말씀드린 건 『데미안』이에요.” |

👉 메모리를 활용하면,  
모델이 “과거 대화의 맥락(Context)”을 유지한 채로  
**자연스러운 멀티턴(Multi-turn) 대화형 챗봇**을 만들 수 있습니다.

### 🔹 LangChain에서의 Memory 개념

LangChain은 다양한 형태의 “기억 클래스(memory classes)”를 제공합니다.  
대표적으로 아래 세 가지를 이해하면 충분합니다 👇

| Memory 클래스 | 설명 | 특징 |
|----------------|------|------|
| **`ConversationBufferMemory`** | 단순히 대화 전체를 계속 저장 | 가장 기본적이고 직관적 |
| **`ConversationBufferWindowMemory`** | 최근 N개의 대화만 저장 | “단기 기억” 형태로, 긴 대화를 효율적으로 유지 |
| **`ConversationSummaryMemory`** | 이전 대화를 LLM이 요약해서 저장 | 긴 대화를 핵심 요약본으로 관리, 맥락 유지에 적합 |

### 🔹 프롬프트에 직접 넣는 방식 vs Memory 클래스 사용 비교

| 비교 항목 | 프롬프트에 직접 삽입 | Memory 클래스 사용 |
|------------|----------------------|---------------------|
| 코드 복잡도 | ❌ 대화마다 프롬프트를 새로 생성해야 함 | ✅ LangChain이 자동으로 이전 대화 삽입 |
| 확장성 | ❌ 대화가 길어지면 토큰 초과 문제 발생 | ✅ 오래된 대화는 요약/제한 가능 |
| 유지보수 | ⚠️ 과거 대화 삽입 위치만 교체하면 되지만, 요약·토큰 제어를 직접 구현해야 함 | ✅ 메모리 객체만 관리하면 됨 |
| 현실성 | ❌ “일회용 챗봇”에만 적합 | ✅ “지속형 대화형 챗봇” 구현 가능 |

즉, **Memory는 LLM이 “대화 히스토리를 스스로 관리”하도록 하는 스마트한 도우미 클래스**입니다.

### 🔹 Memory 작동 원리
1. 사용자가 LLM에 **질문(입력)** 을 보냅니다.  

2. LLM이 **답변**을 생성하면, 이 대화 내용이 자동으로 **Memory에 저장**됩니다.  
3. 사용자가 다음 질문을 입력하면,  
   LangChain이 **이전 대화 기록을 자동으로 프롬프트에 포함시켜** LLM에 전달합니다.  
4. LLM은 이 정보를 바탕으로 **맥락을 이해하고 연속적인 대화**를 생성합니다.

→ 즉, 사용자는 별도로 과거 대화를 넣을 필요 없이, LangChain이 **“대화 기록 → 프롬프트 삽입 → LLM 호출”**  
과정을 자동으로 처리합니다.

### 코드 예시 1. 기본 Buffer Memory (대화 전체 저장)

In [7]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferMemory(return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 5️⃣ 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "그럼 그 근처 맛집은 어디야?"]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})

C:\Users\user\AppData\Local\Temp\ipykernel_32636\2717942320.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 좋아요! 부산은 바다와 먹거리, 문화가 모두 모인 도시라 취향에 따라 즐길 거리가 다양해요. 아래를 참고해 취향에 맞는 코스를 골라보세요.

상황별 추천 장소(주요 포인트)
- 해운대 해수욕장 + 동백섬
  - 왜 가볼까: 부산의 대표 해변. 여름에는 해수욕, 가을·봄엔 해변 산책로와 바다 풍경이 좋습니다.
  - 팁: 해변 산책로를 따라 동백섬까지 걸으면 전망이 훌륭하고, 주변에 카페·맛집 많아요. 지하철 해운대역에서 도보 이동이 편합니다.
- 광안리 해수욕장 + 광안대교 야경
  - 왜 가볼까: 밤에 especially 아름다운 다리 야경과 해변 분위기.
  - 팁: 해질녘이나 야간에 산책하기 좋고, 광안리 주변의 해산물 맛집도 많아요.
- 감천문화마을
  - 왜 가볼까: 알록달록 컬러 벽화 골목과 사진 찍기 좋은 포인트가 많아요.
  - 팁: 골목길이 가팔라 걷기 편한 신발 필수. 골목 곳곳에 체험 가게가 있어요.
- 자갈치시장 + 국제시장
  - 왜 가볼까: 싱싱한 해산물 구경과 먹거리 천국.
  - 팁: 자갈치시장에서 해산물 구입 후 근처 식당에서 즉석 조리도 가능. 국제시장도 함께 돌아보면 좋아요.
- 남포동 BIFF 광장
  - 왜 가볼까: 영화의 거리 느낌의 거리 예술과 맛집·카페 밀집 지역.
  - 팁: 밤에 분위기가 좋고, 근처의 광복로 쇼핑도 함께 즐길 수 있어요.
- 태종대 & 등대길
  - 왜 가볼까: 해안 절벽과 바다 경관, 등대 산책길이 멋져요.
  - 팁: 바람 많이 부는 날 준비물(겉옷)을 챙기고, 안전 표지판을 따라 걷기.
- 오륙도 스카이워크
  - 왜 가볼까: 바다 위 유리바닥 위를 걷는 짜릿한 체험.
  - 팁: 날씨가 맑을 때 가장 좋고, 인근에 카페나 뷰 포인트가 많아요.
- 해동용궁사
  - 왜 가볼까: 바다를 배경으로 한 고즈넉한 사찰 풍경.
  - 팁: 계단이 많아 편한 신발 추천. 해식절벽의 바다 전망이 멋져요.
- 부산타워(용두산공원) + 근처 남포동
  - 왜 가볼까

모델은 앞선 “부산 여행” 대화를 기억하고 자연스럽게 이어서 대답합니다.

In [8]:
memory.load_memory_variables({})['history']

[HumanMessage(content='부산 여행지 추천해줘.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='좋아요! 부산은 바다와 먹거리, 문화가 모두 모인 도시라 취향에 따라 즐길 거리가 다양해요. 아래를 참고해 취향에 맞는 코스를 골라보세요.\n\n상황별 추천 장소(주요 포인트)\n- 해운대 해수욕장 + 동백섬\n  - 왜 가볼까: 부산의 대표 해변. 여름에는 해수욕, 가을·봄엔 해변 산책로와 바다 풍경이 좋습니다.\n  - 팁: 해변 산책로를 따라 동백섬까지 걸으면 전망이 훌륭하고, 주변에 카페·맛집 많아요. 지하철 해운대역에서 도보 이동이 편합니다.\n- 광안리 해수욕장 + 광안대교 야경\n  - 왜 가볼까: 밤에 especially 아름다운 다리 야경과 해변 분위기.\n  - 팁: 해질녘이나 야간에 산책하기 좋고, 광안리 주변의 해산물 맛집도 많아요.\n- 감천문화마을\n  - 왜 가볼까: 알록달록 컬러 벽화 골목과 사진 찍기 좋은 포인트가 많아요.\n  - 팁: 골목길이 가팔라 걷기 편한 신발 필수. 골목 곳곳에 체험 가게가 있어요.\n- 자갈치시장 + 국제시장\n  - 왜 가볼까: 싱싱한 해산물 구경과 먹거리 천국.\n  - 팁: 자갈치시장에서 해산물 구입 후 근처 식당에서 즉석 조리도 가능. 국제시장도 함께 돌아보면 좋아요.\n- 남포동 BIFF 광장\n  - 왜 가볼까: 영화의 거리 느낌의 거리 예술과 맛집·카페 밀집 지역.\n  - 팁: 밤에 분위기가 좋고, 근처의 광복로 쇼핑도 함께 즐길 수 있어요.\n- 태종대 & 등대길\n  - 왜 가볼까: 해안 절벽과 바다 경관, 등대 산책길이 멋져요.\n  - 팁: 바람 많이 부는 날 준비물(겉옷)을 챙기고, 안전 표지판을 따라 걷기.\n- 오륙도 스카이워크\n  - 왜 가볼까: 바다 위 유리바닥 위를 걷는 짜릿한 체험.\n  - 팁: 날씨가 맑을 때 가장 좋고, 인근에 카페나 뷰 포인트가 많아요.\n- 해동용궁사\n  - 

### 코드 예시 2. 최근 대화만 유지 (Window Memory)

In [9]:
### 코드 예시 ① 기본 Buffer Memory
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano")

# 메모리 객체 생성 (대화 전체를 버퍼 형태로 저장)
memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 전문가야. 사용자의 질문에 친절하게 답해줘."),
    MessagesPlaceholder(variable_name="history"),  # 이전 대화 삽입 위치
    ("human", "{input}")                           # 사용자 질문
]) 

# LCEL 체인 구성
chain = prompt | llm 

# 5️⃣ 연속 대화 시뮬레이션
inputs = ["부산 여행지 추천해줘.", "대한 민국의 수도는 어디야?", "서울의 인구는 몇명이야?", "내가 아까 추천해달라고 한 어행지는 어디야?" ]

for user_input in inputs:
    # 메모리 불러오기
    history = memory.load_memory_variables({})["history"]
    # 실행
    result = chain.invoke({"history": history, "input": user_input})
    # 결과 출력
    print(f"\n사용자: {user_input}\n 응답: {result.content}")
    # 메모리에 저장
    memory.save_context({"input": user_input}, {"output": result.content})


C:\Users\user\AppData\Local\Temp\ipykernel_32636\3298101796.py:10: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(k=2, return_messages=True)



사용자: 부산 여행지 추천해줘.
 응답: 좋아요! 부산 여행지 추천을 카테고리별로 정리해봤어요. 취향이나 체류 기간에 따라 선택해보면 좋을 것 같아요.

꼭 가봐야 할 대표 명소
- 해운대 해수욕장 + 동백섬: 도시의 대표 해변. 해변 산책로와 해운대 달맞이길도 함께 즐겨요.
- 광안리 해수욕장 + 광안대교 야경: 야경 명소로 유명한 곳. 저녁 노을과 다리 조명이 멋져요.
- 자갈치 시장 & 국제시장: 신선한 해산물 구경하고 먹거리 골목을 걷기 좋습니다.
- 남포동/BIFF 광장: 영화 축제 분위기와 거리 음식, 쇼핑.
- 감천문화마을: 알록달록 계단와 예술 벽화가 매력적인 마을.
- 태종대 공원: 해안 절벽 산책로와 전망대; 바다 풍경이 시원합니다.
- 해동용궁사: 바다를 배경으로 한 절경 사찰.
- 범어사: 금정산 자락의 큰 사찰로 산책하기 좋습니다.
- 오륙도 스카이워크: 바다 위 전망 데크에서 바다를 한눈에.
- 부산 타워/용두산공원: 도시 파노라마 뷰 포인트.

먹거리 & 분위기 추천
- 자갈치시장: 신선한 해산물 구경과 즉석 해산물 맛보기.
- 국제시장 주변: 길거리 음식과 전통 시장 분위기 체험.
- 부산식 대표 요리 체험: 밀면, 돼지국밥, 해산물 요리 등 현지 맛집 탐방.
- 해변 카페나 레스토랑에서 야경 감상도 좋습니다(광안리 주변 추천).

자연과 산책 명소
- 달맞이길(해운대)과 해변 산책로: 산책하기 좋고 경치 좋음.
- 금정산 등산코스와 범어사: 가벼운 트레킹이나 절경 감상에 좋아요.
- 오륙도 스카이워크: 바다를 아래로 두고 걷는 느낌.

일정 예시 (2~3일)
- 2일 코스
  Day 1: 자갈치시장/BIFF 광장 → 감천문화마을 → 남포동 산책 → 광안리 해변에서 저녁 및 광안대교 야경
  Day 2: 해운대 해변 및 동백섬 산책 → 달맞이길 산책/카페 → 필요시 해운대 인근 맛집
- 3일 코스 보완
  Day 3: 태종대 공원 또는 해동용궁사 방문 → 금정산/범어사 중 한 곳 선택 → 부산역이나 시내로 돌아와 남은 시간 쇼핑


이 방식은 단기 기억처럼 작동합니다.  
오래된 대화는 자동으로 지워져 토큰 낭비를 줄이고 효율성을 높입니다.

### 코드 예시 3. 이전 대화를 요약해서 저장 (ConversationSummaryMemory)

In [10]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 요약형 메모리 생성 — LLM이 과거 대화를 요약해 맥락을 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 여행 플래너야. 사용자의 요구에 따라 가족 여행 일정을 제안해줘."),
    MessagesPlaceholder(variable_name="history"),  # 요약된 과거 대화 자동 삽입
    ("human", "{input}")                           # 현재 질문
])

# LCEL 체인 구성
chain = prompt | llm

# 연속 대화 시뮬레이션
inputs = [
    "이번 주말에 가족 여행지 추천해줘.",
    "지난번에 추천한 곳 중에 아이들이 놀기 좋은 곳은 어디였지?",
    "그럼 거기 일정표를 하루만 짜줘."
]

for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n👤 사용자: {user_input}\n🤖 응답: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

C:\Users\user\AppData\Local\Temp\ipykernel_32636\2842420670.py:9: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)



👤 사용자: 이번 주말에 가족 여행지 추천해줘.
🤖 응답: 좋아요! 이번 주말 가족 여행으로 부담 없이 다녀올 수 있는, 아이들도 좋아하는 장소 위주로 몇 가지를 뽑아봤어요. 위치를 알려주시면 더 맞춤형으로 조정해드릴게요. 우선 수도권 출발 기준으로 인기 있는 4~5곳입니다.

1) 가평 남이섬 + 아침고요수목원(당일 or 1박)
- 왜 좋나요: 아이들이 좋아하는 나무길 산책, 자전거 타기, 동화마을 느낌의 분위기. 부모님도 한적한 자연 속 힐링.
- 예상 소요: 서울에서 차로 약 1.5–2시간
- 간단 일정 예시
  - 토: 출발 → 남이섬 자전거/데이트 코스 → 점심 → 아침고요수목원 slight 코스 → 가평 근처 숙소 1박
  - 일: 수목원에서 느긋하게 산책 후 귀가
- 팁: 자전거 대여 확인, 주말은 교통 체증 주의

2) 에버랜드 + 캐리비안베이(용인, 당일)
- 왜 좋나요: 놀이기구부터 워터파크까지 아이들 취향 모두 커버. 비오는 날에도 실내 공간 커버가 비교적 좋음.
- 예상 소요: 서울에서 차로 약 40–70분
- 간단 일정 예시
  - 토: 이른 출발 → 에버랜드에서 하루 종일 즐김 → 캐리비안베이(선택) → 근처 숙소 1박
  - 일: 가족 맛집 탐방 후 귀가
- 팁: 날씨에 따라 물놀이 의류 준비, 이용권 미리 구매하면 대기시간 단축

3) 속초/설악산 (당일 혹은 1박)
- 왜 좋나요: 자연 체험, 케이블카 타고 정상 뷰 감상, 바다 냄새 맡으며 산책. 아이들도 자연 교과 체험에 좋아요.
- 예상 소요: 서울에서 차로 약 3시간 ~ 3시간 30분
- 간단 일정 예시
  - 토: 출발 → 설악케이블카 체험 → 속초 수산시장 근처 점심 → 속초 해수욕장 또는 속초 해안 산책
  - 1박: 속초 시내 숙박 → 다음 날 속초 중앙시장+청초호/영금정 코스
- 팁: 주차와 날씨 체크 필수, 짧은 코스로 아이 컨디션 관리

4) 강릉 경포대/안목커피거리(당일)
- 왜 좋나요: 바다 산책, 모래놀이 가능, 가족 카페 타임으로 휴식도 쉬움. 가을 바

In [11]:
# 현재까지의 요약본 확인
print(memory.buffer)

New summary:
The human previously asked which of the recommended weekend trips are best for kids to play. The AI responded in Korean with five options (Everland + Caribbean Bay; Nami Island + Morning Calm Arboretum; Gangneung Gyeongpo Beach + Anmok Coffee Street; Sokcho/Seoraksan; Paju Heyri Art Village + Imjingak), outlining why each suits children and offering practical tips, and invited sharing details like children's ages, starting point, budget, and vibe to draft a customized 1-night-2-day itinerary. In the latest exchange, the human asked for a one-day schedule. The AI then provided three representative one-day itineraries, each with a detailed timetable and notes: Option 1 Everland + Caribbean Bay near Seoul (about 9 hours), Option 2 Namis Island + Morning Calm Garden (about 9 hours), and Option 3 Paju Heyri Art Village + Imjingak (about 9 hours). It also included common tips (pace to fit kids, weather prep, rest, and travel time) and asked for departure location, children's age

이 방식은 LLM이 이전 대화를 스스로 요약하여 핵심 맥락을 유지합니다.  
수백 문장의 대화도 핵심 내용만 남기기 때문에 효율적입니다.

**핵심 정리**
| Memory 타입 | 특징 | 사용 목적 |
|--------------|--------|-------------|
| **BufferMemory** | 모든 대화를 그대로 저장 | 짧은 대화, 디버깅용 |
| **WindowMemory** | 최근 N개의 대화만 유지 | 실시간 채팅, 효율성 |
| **SummaryMemory** | 이전 대화를 요약해서 유지 | 긴 대화 맥락 유지형 챗봇 |

🔹 실습 : 아래 조건을 만족하는 “나만의 대화형 여행 상담 챗봇”을 만들어보세요.

당신은 AI 여행 비서 트래블GPT 입니다.  
고객이 여러 도시를 순서대로 여행하면서 맞춤 일정·숙소·음식을 요청할 때,  
이전 대화 내용을 기억하며 연결된 제안을 해주는 지능형 여행 어시스턴트를 구현하세요.  

1. ChatOpenAI(model="gpt-5-nano") 사용
2. ConversationSummaryMemory로 요약 기반 대화 기억 구현
3. 답변에 반드시 "이전 여행 내용을 바탕으로 추천드리면..." 이라는 문구 포함
  - 예시 : 이전 여행 내용을 바탕으로 추천드리면, 여수에서는 해상케이블카와 낭만포차거리를 꼭 가보세요.
4. 대화 시나리오:
  - 사용자가 “부산 → 여수 → 강릉” 순으로 도시를 이동
  - 챗봇은 이전 도시에서 한 활동을 기억하고 “연결된 여행 루트”나 “테마별 추천(가족/커플/힐링)”을 제안할 것

예시 시나리오
```bash
사용자: 이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.
AI: 부산의 해운대, 아쿠아리움이 가족 단위로 인기예요!

사용자: 이번엔 여수로 가볼까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 부산의 해변 감성에 이어 여수에서는 바다 전망 케이블카와 낭만포차를 즐기세요.

사용자: 그럼 마지막은 강릉이 좋을까?
AI: 이전 여행 내용을 바탕으로 추천드리면, 강릉에서는 여수보다 조용한 힐링 카페 거리와 바다 일출 코스를 권합니다.
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 1. 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 2. 요약형 메모리 생성
memory = ConversationSummaryMemory(
    # TODO: 필요한 인자를 채워보세요.
)

# 3. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     # TODO: 트래블GPT의 역할과 반드시 포함할 문구 조건을 작성해보세요.
    ),
    # TODO: 이전 대화(history)가 들어갈 자리를 추가해보세요.
    ("human", "{input}")
])

# 4. LCEL 체인 구성
chain = ...

# 5. 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 6. 연속 대화 실행
for user_input in inputs:
    history = ...
    result = ...
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context(...)

# 7. 선택: 요약된 메모리 확인
print(memory.buffer)

ValueError: Expected 2-tuple of (role, template), got ('system',)

<details>
<summary>정답 보기</summary>

```python
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts  import ChatPromptTemplate, MessagesPlaceholder

# 모델 선언
llm = ChatOpenAI(model="gpt-5-nano", temperature=0.7)

# 메모리 생성 — 이전 대화를 요약하며 맥락 유지
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 여행 비서 트래블GPT야. "
     "사용자의 여행 루트를 기억하고, 이전 여행 내용을 바탕으로 다음 도시를 추천해줘. "
     "답변에는 반드시 '이전 여행 내용을 바탕으로 추천드리면,' 이라는 문구를 포함해야 해."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# LCEL 체인 구성
chain = prompt | llm

# 대화 시나리오
inputs = [
    "이번 주말엔 부산 갈 건데 가족 여행지 좀 추천해줘.",
    "이번엔 여수로 가볼까?",
    "그럼 마지막은 강릉이 좋을까?"
]

# 연속 대화 시뮬레이션
for user_input in inputs:
    history = memory.load_memory_variables({})["history"]
    result = chain.invoke({"history": history, "input": user_input})
    print(f"\n사용자: {user_input}\n트래블GPT: {result.content}")
    memory.save_context({"input": user_input}, {"output": result.content})

# 대화 요약 확인 (선택)
print("\n요약된 Memory Buffer:")
print(memory.buffer)
```
</details>